### PART 1: Wrangling and EDA

Demographic: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/DEMO.htm \
Mortality: https://www.cdc.gov/nchs/data/datalinkage/public-use-linked-mortality-file-description.pdf \
Caffeine: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/SSCAFE_A.htm 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Merge of mortality and demo into 'df'
mdf = pd.read_csv('linked_mortality_file_1999_2000.csv')
gdf = pd.read_sas("DEMO.xpt", format="xport")
df = gdf.merge(mdf, on="SEQN", how="inner")


### Description of Variables 

* ELIGSTAT (Eligibility Status for Mortality Follow-up): This variable shows whether or not a person is defined as eligible for mortality linkage due to insufficient identifying data. 
* MORTSTAT (Final Mortality Status): This variable determines the vital status of a participant and should be used when determining survival rates. Participants assumed alive are numbered 0 while while those assumed deceased are numbered 1. 
* PERMTH_INT (Number of Person Months of Follow-Up from NHANES Interview Date): This variable describes the number of person-months of the follow up from the interview date. 
* RIDAGEEX: This variable describes the age in months at the time the participant took the MEC exam. This data is only provided for those who were less than 85 years old at the time of the household screening interview. 

In [ ]:
# Merge of df and caffeine dataset
cdf = pd.read_sas("SSCAFE_A.xpt", format="xport")
merged_df = cdf.merge(df, on="SEQN", how="inner")

**ANSWER** Explain rationale for choosing this dataset

**ANSWER** Document any missing values here

In [ ]:
# Choosing variables of interest
cols = ["SEQN", "ELIGSTAT", "MORTSTAT", "PERMTH_INT", "RIDAGEEX", "RIAGENDR",
        "WTSSCAF2", "WTSSCAF4", "SSMX1", "SSMX1LC","SSMX2", "SSMX2LC","SSMX3", "SSMX3LC",
        "SSMX4", "SSMX4LC", "SSMX5", "SSMX5LC", "SSMX6", "SSMX6LC",
        "SSMX7", "SSMX7LC"]
new_df = merged_df[cols].copy()
new_df.to_csv("working_df.csv", index=False)
pd.set_option('display.max_columns', None)
new_df.head()


### EDA and visualization bomboclat blow

In [ ]:
# copy paste of stuff yas
print("h")

Based off EDA
**ANSWER**
1) Are any important variables skewed? 
2) Are there outliers? 
3) How correlated are pairs of variables? 
4) Do pairs of categorical variables exhibit interesting patterns in contingency tables? 
5) Provide a clear discussion and examination of the data and the variables you are interested in using

### Part 2: KNN Classification/Regression

1) Describe the data, particularly what an observation is and whether there are any missing data that might impact your analysis. Who collected the data and why? What known limitations are there to analysis? (10/100 pts)\
**The dataset is made up of a few important columns, namely the sequence number (SEQN), the eligibility status (ELIGSTAT), the death status (MORTSTAT). It also includes the demographic variables such as age (in months) and gender (1 = male, 2 = female). Included is also a few variables from the caffeine dataset including the representative weight (we are using WTSSCAF2 since we are looking at 1 cycle from 1999-2000), and several metabolites SSMX1-7 along wit their LC (lab comment version). We are looking at SSMX1 specifically because it represents how much caffeine an individual was exposed to. We ignore the LC version during regression**\
**There are sme missing data. Particularly, there are 942 missing data from the mortality status column out of 2648, which need to be removed before we can regress on it. There are also 53 missing ages, which isn't particularly important so we can either keep or remove it. In addition, for each of the caffeine metabolites, there are 592 values missing out of 2648. Since we are looking at the relationship between SSMX1 and death, we need to drop these NA's.** \
**The data was collected by NHANES to see if there is a broader implicaton of consumption of caffeine on the health of the general public. Caffeine is so accessible and many people consume it daily. The data is collected through random sampling and use of physical exams and urine samples in this dataset.**\
**There are several limitations. First, we are looking at caffeine intake at a singular point in time, not over the course of a long time. Therefore, we may not be able to say that long-term habitual caffeine consumption is linked to death. In addition, we do not take into consideration any confounding variables like presdispositions, prior health issues, and anything beyond age and sex.**


In [ ]:
print(new_df.shape)
new_df.isna().sum()

In [ ]:
new_df.describe()

2) Natalie

1. Using your variables to predict mortality using a k-Nearest Neighbor Classifier. Analyze its performance and explain clearly how you select k.

In [ ]:
#We used the three variables SSMX3, SSMX4, SSMX5 to predict MORTSTAT.
#First, import relevant libraries and modules. 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

#Next, update and load cleaned dataset and keep relevant columns
merged_df = pd.read_csv("merged_demo_mortality_caffeine.csv")
cols = ["SEQN", "ELIGSTAT", "MORTSTAT", "PERMTH_INT", "RIDAGEEX", "RIAGENDR",
        "WTSSCAF2", "WTSSCAF4",
        "SSMX3", "SSMX3LC", "SSMX4", "SSMX4LC", "SSMX5", "SSMX5LC",
        "SSMX6", "SSMX6LC", "SSMX7", "SSMX7LC"]

working_df = merged_df[cols].copy()
working_df.to_csv("working_df.csv", index=False)
df_q3 = working_df[["MORTSTAT", "SSMX3", "SSMX4", "SSMX5"]].dropna()

In [ ]:
X = df_q3[["SSMX3", "SSMX4", "SSMX5"]]
y = df_q3["MORTSTAT"].astype(int)

#Split the data into training and testing (80% vs 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#standardized the variables to prevent bias in distance-based algorithms like KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Testing k values between 1 and 20 and selecting the highest accuracy to determine the best k for KNN
k_values = range(1, 21)
accuracies = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
best_k = k_values[np.argmax(accuracies)]
print("Best k:", best_k)

In [ ]:
plt.plot(k_values, accuracies)
plt.xlabel("k")
plt.ylabel("Test Accuracy")
plt.title("Accuracy vs k")
plt.show()
#This graph shows which k value gives the highest accuracy, which is 8 in this case.

In [ ]:
#Fit the final kNN model using the selected best value of k 
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)
#Finally, generate predictions on the test set and evaluate the final model's performance
y_final_pred = final_model.predict(X_test_scaled)

print("Final Accuracy:", accuracy_score(y_test, y_final_pred))

To predict MORTSTAT, I used a k-Nearest classifier with the variables SSMX3, SSMX4, and SSMX5 as predictors. After cleaning and removing missing variables and keeping only relevant columns, I split the data into 80% training and 20% testing. Additionally, I standardized the predictors before fitting the model.

Next, to select k, I tested values from 1 to 20 and calculated the testing accuracy for each value, as done on the homework. Finally, upon looking at the graph and calculations, I chose the k that produced the highest test accuracy. This is important as a small k value overfits data, while large k values tend to overfit. Selecting a k with the highest test accuracy reduces bias and variance in the data.

Finally, using the optimal k, I created the final model with a test accuracy of 0.7121771217712177.

4. Ashley

5. Angel

Describe how your model could be used for health interventions based on patient characteristics. Are there any limitations or risks to consider?

Our model can take someone’s basic info like age (RIDAGEEX) and sex (RIAGENDR), along with their urine caffeine metabolite levels, and estimate their mortality risk over the NHANES follow-up period. We used SSMX3 (7-methylxanthine), SSMX4 (theophylline), and SSMX5 (paraxanthine), since they’re all measurable breakdown products related to caffeine exposure. In a clinic or public health setting, this kind of estimate could help staff decide who might need a little more follow-up first, like a quick conversation about caffeine habits, checking related issues such as sleep and blood pressure, or recommending lifestyle changes and monitoring sooner.

A few limits matter. This model shows correlation, not causation, so a higher predicted risk does not mean caffeine is the direct cause. Things like smoking, existing disease, medications, and socioeconomic factors could be part of what the model is picking up. The model might also work better for some groups than others, such as certain ages or sexes, which could lead to less accurate guidance if you do not check for that. Finally, because the data comes from NHANES 1999–2000, it may not fully match today’s caffeine products and health patterns, so it would need testing before being used in modern settings.